# WavqWise: Trading Forecast & Technical Analysis
**Sense. Forecast. Alert.**

Forecast stock prices, add technical indicators, and generate trading signals.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VK-Ant/wavqwise/blob/main/demos/notebooks/wavqwise_trading_forecast.ipynb)

**Author:** [VK-Ant](https://github.com/VK-Ant)

In [ ]:
!pip install wavqwise -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wavqwise import WavqPipeline
from wavqwise.trading.indicators.momentum import RSIIndicator
from wavqwise.trading.indicators.trend import MACDIndicator, SMAIndicator
from wavqwise.trading.indicators.volatility import BollingerBandsIndicator
print('WavqWise loaded')

## 1. Generate Realistic Stock Data

In [ ]:
def generate_stock(days=500, seed=42):
    np.random.seed(seed)
    dates = pd.date_range('2023-01-02', periods=days, freq='B')
    price, prices, regime = 100.0, [100.0], 0
    for i in range(days-1):
        if np.random.random() < 0.02: regime = np.random.choice([0,1,2])
        drift = {0:0.0002, 1:0.001, 2:-0.0008}[regime]
        vol = {0:0.015, 1:0.012, 2:0.022}[regime]
        price *= (1 + np.random.normal(drift, vol))
        prices.append(max(price, 1))
    prices = np.array(prices)
    df = pd.DataFrame({'Date':dates, 'Open':prices*(1+np.random.uniform(-0.005,0.005,days)),
        'High':prices*(1+np.random.uniform(0.002,0.025,days)),
        'Low':prices*(1-np.random.uniform(0.002,0.025,days)),
        'Close':prices, 'Volume':(np.random.lognormal(15,0.5,days)).astype(int)})
    df['High'] = df[['Open','High','Close']].max(axis=1)
    df['Low'] = df[['Open','Low','Close']].min(axis=1)
    return df

stock = generate_stock()
print(f'Period: {stock["Date"].iloc[0].date()} to {stock["Date"].iloc[-1].date()}')
print(f'Return: {(stock["Close"].iloc[-1]/stock["Close"].iloc[0]-1)*100:.1f}%')
stock.tail()

## 2. Add Technical Indicators

In [ ]:
stock = RSIIndicator(14).compute(stock)
stock = MACDIndicator().compute(stock)
stock = SMAIndicator(20).compute(stock)
stock = SMAIndicator(50).compute(stock)
stock = BollingerBandsIndicator(20, 2).compute(stock)

print(f'RSI: {stock["RSI"].iloc[-1]:.1f}')
print(f'MACD: {stock["MACD"].iloc[-1]:.4f}')
print(f'SMA-20 vs SMA-50: {"Bullish" if stock["SMA_20"].iloc[-1] > stock["SMA_50"].iloc[-1] else "Bearish"}')

## 3. Forecast with WavqPipeline

In [ ]:
pipeline = WavqPipeline()
pipeline.load(stock, target='Close', time='Date')

# Forecast 30 trading days
forecast_sma = pipeline.forecast(horizon=30, model='moving_average')
forecast_ema = pipeline.forecast(horizon=30, model='ema')

print(f'SMA forecast (day 1): ${forecast_sma.forecast["Close"].iloc[0]:.2f}')
print(f'EMA forecast (day 1): ${forecast_ema.forecast["Close"].iloc[0]:.2f}')

## 4. Compare Models

In [ ]:
comparison = pipeline.compare_models(['moving_average','ema','naive','seasonal_naive'], horizon=14)
print(comparison.to_string(index=False))

## 5. Full Trading Chart

In [ ]:
clean = stock.dropna().copy()
fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [3, 1, 1]})

# Price + Bollinger + Forecast
ax = axes[0]
ax.plot(clean['Date'], clean['Close'], color='#1e293b', linewidth=1.2, label='Close')
ax.plot(clean['Date'], clean['SMA_20'], color='#2563eb', linewidth=0.8, alpha=0.7, label='SMA-20')
ax.plot(clean['Date'], clean['SMA_50'], color='#dc2626', linewidth=0.8, alpha=0.7, label='SMA-50')
ax.fill_between(clean['Date'], clean['BB_lower'], clean['BB_upper'], alpha=0.1, color='#6366f1')
ax.plot(forecast_sma.forecast['Date'], forecast_sma.forecast['Close'], '--', color='#059669', linewidth=2, label='Forecast')
ax.fill_between(forecast_sma.forecast['Date'], forecast_sma.forecast['Close_lower'], forecast_sma.forecast['Close_upper'], alpha=0.15, color='#059669')
ax.legend(loc='upper left', fontsize=8)
ax.set_ylabel('Price ($)'); ax.grid(True, alpha=0.3)

# RSI
ax = axes[1]
ax.plot(clean['Date'], clean['RSI'], color='#7c3aed', linewidth=1)
ax.axhline(70, color='#dc2626', linestyle='--', alpha=0.5)
ax.axhline(30, color='#059669', linestyle='--', alpha=0.5)
ax.set_ylabel('RSI'); ax.set_ylim(0,100); ax.grid(True, alpha=0.3)

# MACD
ax = axes[2]
ax.plot(clean['Date'], clean['MACD'], color='#2563eb', linewidth=1, label='MACD')
ax.plot(clean['Date'], clean['MACD_signal'], color='#dc2626', linewidth=1, label='Signal')
colors = ['#059669' if v>=0 else '#dc2626' for v in clean['MACD_hist']]
ax.bar(clean['Date'], clean['MACD_hist'], color=colors, alpha=0.5, width=1)
ax.legend(fontsize=8); ax.set_ylabel('MACD'); ax.grid(True, alpha=0.3)

plt.suptitle('WavqWise Trading Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
**WavqWise** - Sense. Forecast. Alert. | [GitHub](https://github.com/VK-Ant/wavqwise) | [PyPI](https://pypi.org/project/wavqwise/)